In [1]:
print("Hello Andrii")

Hello Andrii


In [3]:
print("Test")

Test


In [2]:
import pandas as pd
df = pd.read_csv(
    "data/styles.csv",
    on_bad_lines="skip"
)
df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [4]:
df.tail()
df.info()
df.describe()
df.isnull().sum()
df.isnull().sum() / len(df)
df.isnull().sum() / len(df)
df.isnull().sum() / len(df)

<class 'pandas.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  str    
 2   masterCategory      44424 non-null  str    
 3   subCategory         44424 non-null  str    
 4   articleType         44424 non-null  str    
 5   baseColour          44409 non-null  str    
 6   season              44403 non-null  str    
 7   year                44423 non-null  float64
 8   usage               44107 non-null  str    
 9   productDisplayName  44417 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 3.4 MB


id                    0.000000
gender                0.000000
masterCategory        0.000000
subCategory           0.000000
articleType           0.000000
baseColour            0.000338
season                0.000473
year                  0.000023
usage                 0.007136
productDisplayName    0.000158
dtype: float64

In [5]:
df.baseColour.value_counts()

baseColour
Black                9728
White                5538
Blue                 4918
Brown                3494
Grey                 2741
Red                  2455
Green                2115
Pink                 1860
Navy Blue            1789
Purple               1640
Silver               1090
Yellow                778
Beige                 749
Gold                  628
Maroon                581
Orange                530
Olive                 410
Multi                 394
Cream                 390
Steel                 315
Charcoal              228
Peach                 195
Off White             182
Skin                  179
Lavender              162
Grey Melange          146
Khaki                 139
Magenta               129
Teal                  120
Tan                   114
Mustard                97
Bronze                 95
Copper                 86
Turquoise Blue         69
Rust                   66
Burgundy               45
Metallic               43
Coffee Brown           31
M

In [6]:
# Leave only relevant columns
df = df[["id", "gender", "masterCategory", "baseColour"]]

# Drop missing values
df = df.dropna()

df.head()

,id,gender,masterCategory,baseColour
0,15970,Men,Apparel,Navy Blue
1,39386,Men,Apparel,Blue
2,59263,Women,Accessories,Silver
3,21379,Men,Apparel,Black
4,53759,Men,Apparel,Grey


In [11]:
import os

# styles.csv has no image path column; files follow data/images/<id>.jpg (same layout as styles under data/).
_image_dir = os.path.join("data", "images")
df = df[df["id"].map(lambda i: os.path.exists(os.path.join(_image_dir, f"{int(i)}.jpg")))]

In [16]:
IMAGE_DIR = "data/images" 

df["image_path"] = df["id"].astype(str) + ".jpg"
df["image_path"] = df["image_path"].apply(lambda x: os.path.join(IMAGE_DIR, x))

In [18]:
df.head()

,id,gender,masterCategory,baseColour,image_path
0,15970,Men,Apparel,Navy Blue,data/images/15970.jpg
1,39386,Men,Apparel,Blue,data/images/39386.jpg
2,59263,Women,Accessories,Silver,data/images/59263.jpg
3,21379,Men,Apparel,Black,data/images/21379.jpg
4,53759,Men,Apparel,Grey,data/images/53759.jpg


Start Model

In [12]:
from torch.utils.data import Dataset
from PIL import Image

class FashionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        label = {
            "category": row["masterCategory"],
            "gender": row["gender"],
            "color": row["baseColour"]
        }

        if self.transform:
            image = self.transform(image)

        return image, label

In [14]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [19]:
dataset = FashionDataset(df, transform=transform)

image, label = dataset[0]

print(label)

{'category': 'Apparel', 'gender': 'Men', 'color': 'Navy Blue'}


Map colors to FitFolio colors

In [27]:
# Every `baseColour` in styles.csv (46 labels). Unmapped e.g. NA → NaN in color_12.
COLOR_MAP = {
    "Beige": "beige",
    "Black": "black",
    "Blue": "blue",
    "Bronze": "brown",
    "Brown": "brown",
    "Burgundy": "red",
    "Charcoal": "grey",
    "Coffee Brown": "brown",
    "Copper": "orange",
    "Coral": "orange",
    "Cream": "beige",
    "Fluorescent Green": "green",
    "Gold": "yellow",
    "Gray": "grey",
    "Green": "green",
    "Grey": "grey",
    "Grey Melange": "grey",
    "Khaki": "beige",
    "Lavender": "purple",
    "Lime Green": "green",
    "Magenta": "pink",
    "Maroon": "red",
    "Mauve": "purple",
    "Metallic": "grey",
    "Multi": "grey",
    "Mustard": "yellow",
    "Mushroom Brown": "beige",
    "Navy Blue": "blue",
    "Nude": "beige",
    "Olive": "green",
    "Off White": "beige",
    "Orange": "orange",
    "Peach": "orange",
    "Pink": "pink",
    "Purple": "purple",
    "Red": "red",
    "Rose": "pink",
    "Rust": "red",
    "Sea Green": "green",
    "Silver": "grey",
    "Skin": "beige",
    "Steel": "grey",
    "Tan": "brown",
    "Taupe": "beige",
    "Teal": "blue",
    "Turquoise Blue": "blue",
    "Violet": "purple",
    "White": "white",
    "Yellow": "yellow",
}

In [28]:
df["color_12"] = df["baseColour"].map(COLOR_MAP)

In [29]:
df.head()

,id,gender,masterCategory,baseColour,image_path,color_12
0,15970,Men,Apparel,Navy Blue,data/images/15970.jpg,blue
1,39386,Men,Apparel,Blue,data/images/39386.jpg,blue
2,59263,Women,Accessories,Silver,data/images/59263.jpg,grey
3,21379,Men,Apparel,Black,data/images/21379.jpg,black
4,53759,Men,Apparel,Grey,data/images/53759.jpg,grey


In [30]:
df["color_12"].isnull().sum()
df["color_12"].value_counts()
df.head()
df.tail()
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
Index: 44404 entries, 0 to 44423
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id              44404 non-null  int64
 1   gender          44404 non-null  str  
 2   masterCategory  44404 non-null  str  
 3   baseColour      44404 non-null  str  
 4   image_path      44404 non-null  str  
 5   color_12        44404 non-null  str  
dtypes: int64(1), str(5)
memory usage: 2.4 MB


id                0
gender            0
masterCategory    0
baseColour        0
image_path        0
color_12          0
dtype: int64

Other columns

In [31]:
df = df[df["masterCategory"].isin(["Apparel", "Footwear", "Accessories"])]

ALLOWED_GENDERS = ["Men", "Women", "Unisex"]
df = df[df["gender"].isin(ALLOWED_GENDERS)]

In [32]:
print(df["masterCategory"].value_counts())
print(df["gender"].value_counts())
print(df["color_12"].value_counts())

masterCategory
Apparel        20060
Accessories    11229
Footwear        9105
Name: count, dtype: int64
gender
Men       21504
Women     16779
Unisex     2111
Name: count, dtype: int64
color_12
black     9314
blue      6316
white     5039
grey      4760
brown     3425
red       2731
green     2296
purple    1613
pink      1505
beige     1460
yellow    1278
orange     657
Name: count, dtype: int64


In [ ]:
# to match FitFolio

# 1) Rename category value
df["masterCategory"] = df["masterCategory"].replace({
    "Footwear": "Shoes"
})

# 2) Lowercase genders
df["gender"] = df["gender"].str.lower()

In [35]:
print(df["masterCategory"].value_counts())
print(df["gender"].value_counts())
print(df["color_12"].value_counts())

masterCategory
Apparel        20060
Accessories    11229
Shoes           9105
Name: count, dtype: int64
gender
men       21504
women     16779
unisex     2111
Name: count, dtype: int64
color_12
black     9314
blue      6316
white     5039
grey      4760
brown     3425
red       2731
green     2296
purple    1613
pink      1505
beige     1460
yellow    1278
orange     657
Name: count, dtype: int64


In [36]:
print(df["masterCategory"].value_counts(normalize=True))
print(df["gender"].value_counts(normalize=True))
print(df["color_12"].value_counts(normalize=True))

masterCategory
Apparel        0.496608
Accessories    0.277987
Shoes          0.225405
Name: proportion, dtype: float64
gender
men       0.532356
women     0.415383
unisex    0.052260
Name: proportion, dtype: float64
color_12
black     0.230579
blue      0.156360
white     0.124746
grey      0.117839
brown     0.084790
red       0.067609
green     0.056840
purple    0.039932
pink      0.037258
beige     0.036144
yellow    0.031638
orange    0.016265
Name: proportion, dtype: float64


In [37]:
from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
gender_encoder = LabelEncoder()
color_encoder = LabelEncoder()

df["category_label"] = category_encoder.fit_transform(df["masterCategory"])
df["gender_label"] = gender_encoder.fit_transform(df["gender"])
df["color_label"] = color_encoder.fit_transform(df["color_12"])

print("Category classes:", list(category_encoder.classes_))
print("Gender classes:", list(gender_encoder.classes_))
print("Color classes:", list(color_encoder.classes_))

Category classes: ['Accessories', 'Apparel', 'Shoes']
Gender classes: ['men', 'unisex', 'women']
Color classes: ['beige', 'black', 'blue', 'brown', 'green', 'grey', 'orange', 'pink', 'purple', 'red', 'white', 'yellow']


In [38]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df[["category_label"]])

print(train_df.shape, val_df.shape)

(32315, 9) (8079, 9)


In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class FashionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        labels = {
            "category": int(row["category_label"]),
            "gender": int(row["gender_label"]),
            "color": int(row["color_label"]),
        }

        if self.transform:
            image = self.transform(image)

        return image, labels

In [40]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [41]:
from torch.utils.data import DataLoader

train_dataset = FashionDataset(train_df, transform=train_transform)
val_dataset = FashionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [42]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels["category"][:5])
print(labels["gender"][:5])
print(labels["color"][:5])

torch.Size([32, 3, 224, 224])
tensor([2, 1, 1, 0, 1])
tensor([0, 0, 0, 0, 0])
tensor([1, 9, 9, 1, 2])


### Train Model V1

In [ ]:
# improve transforms 
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [44]:
# recreate datasets/loaders
train_dataset = FashionDataset(train_df, transform=train_transform)
val_dataset = FashionDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [45]:
# multi-head model
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class MultiTaskResNet(nn.Module):
    def __init__(self, num_categories, num_genders, num_colors):
        super().__init__()

        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone

        self.category_head = nn.Linear(in_features, num_categories)
        self.gender_head = nn.Linear(in_features, num_genders)
        self.color_head = nn.Linear(in_features, num_colors)

    def forward(self, x):
        features = self.backbone(x)

        return {
            "category": self.category_head(features),
            "gender": self.gender_head(features),
            "color": self.color_head(features),
        }

In [46]:
# create model
num_categories = len(category_encoder.classes_)
num_genders = len(gender_encoder.classes_)
num_colors = len(color_encoder.classes_)

model = MultiTaskResNet(
    num_categories=num_categories,
    num_genders=num_genders,
    num_colors=num_colors
)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

print(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/akudenko/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 17.8MB/s]

mps


In [ ]:
# loss & optimizer (CrossEntropyLoss)
criterion_category = nn.CrossEntropyLoss()
criterion_gender = nn.CrossEntropyLoss()
criterion_color = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [48]:
# one training epoch function
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0

    for images, labels in loader:
        images = images.to(device)
        category_targets = labels["category"].to(device)
        gender_targets = labels["gender"].to(device)
        color_targets = labels["color"].to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss_category = criterion_category(outputs["category"], category_targets)
        loss_gender = criterion_gender(outputs["gender"], gender_targets)
        loss_color = criterion_color(outputs["color"], color_targets)

        loss = loss_category + loss_gender + loss_color
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [49]:
# validation function
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0

    correct_category = 0
    correct_gender = 0
    correct_color = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            category_targets = labels["category"].to(device)
            gender_targets = labels["gender"].to(device)
            color_targets = labels["color"].to(device)

            outputs = model(images)

            loss_category = criterion_category(outputs["category"], category_targets)
            loss_gender = criterion_gender(outputs["gender"], gender_targets)
            loss_color = criterion_color(outputs["color"], color_targets)

            loss = loss_category + loss_gender + loss_color
            total_loss += loss.item()

            pred_category = outputs["category"].argmax(dim=1)
            pred_gender = outputs["gender"].argmax(dim=1)
            pred_color = outputs["color"].argmax(dim=1)

            correct_category += (pred_category == category_targets).sum().item()
            correct_gender += (pred_gender == gender_targets).sum().item()
            correct_color += (pred_color == color_targets).sum().item()
            total += images.size(0)

    return {
        "loss": total_loss / len(loader),
        "category_acc": correct_category / total,
        "gender_acc": correct_gender / total,
        "color_acc": correct_color / total,
    }

In [50]:
# train for new epochs
num_epochs = 5

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    print(f"Val Category Acc: {val_metrics['category_acc']:.4f}")
    print(f"Val Gender Acc: {val_metrics['gender_acc']:.4f}")
    print(f"Val Color Acc: {val_metrics['color_acc']:.4f}")
    print("-" * 40)

Epoch 1/5
Train Loss: 1.3243
Val Loss: 1.0591
Val Category Acc: 0.9941
Val Gender Acc: 0.9109
Val Color Acc: 0.7373
----------------------------------------
Epoch 2/5
Train Loss: 0.9714
Val Loss: 0.9822
Val Category Acc: 0.9962
Val Gender Acc: 0.9264
Val Color Acc: 0.7474
----------------------------------------
Epoch 3/5
Train Loss: 0.8278
Val Loss: 0.9882
Val Category Acc: 0.9950
Val Gender Acc: 0.9261
Val Color Acc: 0.7538
----------------------------------------
Epoch 4/5
Train Loss: 0.6952
Val Loss: 0.9605
Val Category Acc: 0.9960
Val Gender Acc: 0.9293
Val Color Acc: 0.7597
----------------------------------------
Epoch 5/5
Train Loss: 0.5900
Val Loss: 0.9901
Val Category Acc: 0.9962
Val Gender Acc: 0.9293
Val Color Acc: 0.7661
----------------------------------------


### Evalutation

In [51]:
from sklearn.metrics import classification_report, confusion_matrix

def get_predictions(model, loader, device, task_name):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            targets = labels[task_name].to(device)

            outputs = model(images)
            preds = outputs[task_name].argmax(dim=1)

            y_true.extend(targets.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return y_true, y_pred

In [53]:
# Color evaluation
y_true_color, y_pred_color = get_predictions(model, val_loader, device, "color")

print(confusion_matrix(y_true_color, y_pred_color))
print(classification_report(
    y_true_color,
    y_pred_color,
    target_names=list(color_encoder.classes_)
))

[[ 189    6    4   41    5   16    2    5    0    5   26    6]
 [   3 1633   58   50    9   96    3    6   10   10   18    9]
 [   2   61 1084    7   17   59    2    1   16   12   27    2]
 [  37   42    8  500    2   24    3    1    4    8    6    7]
 [   5   23   54   17  295   25    0    2    1    2    7    8]
 [  16  124   32   34   17  583    2   12    5   12   97   18]
 [  11    1    1   11    1    4   53   20    0   23    6    5]
 [   5    2    1    5    0    6    0  233   23   32    8    0]
 [   2   11   21    7    0    5    1   15  239   10    8    1]
 [   6    7    2   24    1    9    2   25   13  435   10    3]
 [  40   17   41    6    4   70    0   16    8   14  769    9]
 [  16    2    2   18    7    9    2    3    0    4    5  176]]
              precision    recall  f1-score   support

       beige       0.57      0.62      0.59       305
       black       0.85      0.86      0.85      1905
        blue       0.83      0.84      0.83      1290
       brown       0.69   

In [54]:
y_true_cat, y_pred_cat = get_predictions(model, val_loader, device, "category")
print(classification_report(
    y_true_cat,
    y_pred_cat,
    target_names=list(category_encoder.classes_)
))

              precision    recall  f1-score   support

 Accessories       1.00      0.99      0.99      2246
     Apparel       1.00      1.00      1.00      4012
       Shoes       1.00      1.00      1.00      1821

    accuracy                           1.00      8079
   macro avg       1.00      1.00      1.00      8079
weighted avg       1.00      1.00      1.00      8079



In [55]:
y_true_gender, y_pred_gender = get_predictions(model, val_loader, device, "gender")
print(classification_report(
    y_true_gender,
    y_pred_gender,
    target_names=list(gender_encoder.classes_)
))

              precision    recall  f1-score   support

         men       0.91      0.97      0.94      4278
      unisex       0.78      0.59      0.67       444
       women       0.97      0.92      0.94      3357

    accuracy                           0.93      8079
   macro avg       0.89      0.83      0.85      8079
weighted avg       0.93      0.93      0.93      8079



### Report.
Overall good, problems with:
1. unisex gender, only ~400 samples, difficult to identify. 0.67 F1 and 0.59 recall
2. color, 0.77 accuracy, some strong classes, but some weak. Beige (0.59), grey (0.63), orange (0.51)
3. confusion matrix shows some colors are getting misidentified; beige <-> brown <-> white; grey <-> black; orange <-> red

### Fix Inconsistencies

In [ ]:
# try improving training by adding stonger augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ^ might be crucial for color prediction

In [60]:
# fix class imbalance
from collections import Counter
import torch

counts = Counter(train_df["color_label"])
weights = [1.0 / counts[i] for i in range(len(counts))]
weights = torch.tensor(weights).to(device)

criterion_color = nn.CrossEntropyLoss(weight=weights)

In [ ]:
# try add considence treshold
# probs = torch.softmax(outputs["color"], dim=1)
# confidence, pred = torch.max(probs, dim=1)

# if confidence < 0.6:
#     return "unknown"

In [61]:
# train model with updated transforms and class imbalance
num_epochs = 3

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    print(f"Val Category Acc: {val_metrics['category_acc']:.4f}")
    print(f"Val Gender Acc: {val_metrics['gender_acc']:.4f}")
    print(f"Val Color Acc: {val_metrics['color_acc']:.4f}")
    print("-" * 40)

Epoch 1/3
Train Loss: 0.5835
Val Loss: 1.1664
Val Category Acc: 0.9970
Val Gender Acc: 0.9319
Val Color Acc: 0.7437
----------------------------------------
Epoch 2/3
Train Loss: 0.4409
Val Loss: 1.2924
Val Category Acc: 0.9957
Val Gender Acc: 0.9292
Val Color Acc: 0.7370
----------------------------------------
Epoch 3/3
Train Loss: 0.3619
Val Loss: 1.3657
Val Category Acc: 0.9962
Val Gender Acc: 0.9282
Val Color Acc: 0.7357
----------------------------------------


ResNet-50
EfficientNet
MobileNet if you care about lighter inference